In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os 
import sys 
import warnings
sys.path.append('..')
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt 
import numpy as np
import torch
import compress_pickle
from pathlib import Path

import hippocampalseq as hse
import hippocampalseq.utils as hseu
import hippocampalseq.preprocessing as hsep
import hippocampalseq.analysis as hsea
import hippocampalseq.models as hsem
import hippocampalseq.plotting as hsepl

In [ ]:
def load_model_type(
    data_root: str|Path,
    model_type: str,
    track_type: str,
    groundtruth_file_name: str|Path,
    model_file_name: str|Path,
    rat_name: str|None = None,
) -> dict[str, dict]:
    """Load one model type for tracks of a given type.

    Expected directory layout:
        data_root/
            RatName/
                TrackType1/
                    model_data/
                        <compressed pickle file>
                    preprocessed_data/
                TrackType2/
                    ...

    Args:
        data_root: Directory containing the rat directories.
        model_type: Key to retrieve from the model-data dictionary.
        track_type: Track type prefix, such as "OpenField" or "LinearTrack".
        rat_name: If given, load only this rat. Otherwise load all rats.

    Returns:
        Nested dictionary: result[rat_name][track_directory_name] = model value.
    """
    root = Path(data_root)

    if rat_name is not None:
        rat_dirs = [root / rat_name]
        if not rat_dirs[0].is_dir():
            raise FileNotFoundError(f"Rat directory not found: {rat_dirs[0]}")
    else:
        rat_dirs = sorted(path for path in root.iterdir() if path.is_dir())

    result = {}

    for rat_dir in rat_dirs:
        track_dirs = sorted(
            path
            for path in rat_dir.iterdir()
            if path.is_dir() and path.name.startswith(track_type)
        )

        for track_dir in track_dirs:
            model_file = track_dir / model_file_name
            gt_file = track_dir / groundtruth_file_name
            model_data = compress_pickle.load(model_file)
            ground_truth_data = compress_pickle.load(gt_file)

            result.setdefault(rat_dir.name, {})[track_dir.name] = (
                model_data[model_type],
                ground_truth_data,
                model_data['split_indices'] 
            )

    return result

def run_peak_posteriors(session_data, model_type: str, bin_size: int =2):
    assert model_type in ["MAP", "Gaussian"]
    outputs = {}
    for rat,sessions in session_data.items():
        for session,(model_data,ground_truth,indices) in sessions.items():
            environment_size = ground_truth['raw_data'].environment_size
            theta_data = ground_truth['theta_data']
            if model_type == "Gaussian":
                smoothed_means = model_data['results']['training'].smoothed_mean \
                    + model_data['results']['validation'].smoothed_mean
                smoothed_covs = model_data['results']['training'].smoothed_cov \
                    + model_data['results']['training'].smoothed_cov
                true_position = [
                    theta_data.ground_truth[i] for i in indices['train']
                ] + [
                    theta_data.ground_truth[i] for i in indices['validation']
                ]
                spikes = [
                    theta_data.spikes[i] for i in indices['train']
                ] + [
                    theta_data.spikes[i] for i in indices['validation']
                ]
                spike_count_percentile = None               
                normalized_posterior = hsea.gaussians_to_grids(
                    smoothed_means,
                    smoothed_covs,
                    environment_size,
                    bin_size,
                    model_data['model']['marginal_position_dims']
                )
            else:
                normalized_posterior = hsea.bayesian_map_to_grids(
                    model_data['results']['training'].posterior_probabilities
                ) 
                true_position = theta_data.ground_truth
                spikes = theta_data.spikes
                spike_count_percentile = 66.67

            if "Linear" in session:
                peak_posterior = hsea.peak_posterior(
                    normalized_posterior,
                    environment_size,
                    true_position,
                    spikes,
                    bin_size_cm=bin_size,
                    spike_count_percentile=spike_count_percentile
                )
            else:
                peak_posterior = hsea.peak_posterior_2d(
                    normalized_posterior,
                    environment_size,
                    true_position,
                    spikes,
                    bin_size_cm=bin_size,
                    spike_count_percentile=spike_count_percentile
                )
            outputs.setdefault(rat, {}).setdefault(session[:-1], []).append(
                peak_posterior
            )
    return outputs

def pool_posteriors(peak_posterior, track_type: str = "Linear", pool="rats"):
    def norm(a):
        col = a.sum(axis=0, keepdims=True)
        return a / np.where(col == 0, 1.0, col)

    if pool == "rats":
        all_sessions = []
        for sessions in peak_posterior.values():
            all_sessions += [session['dist_of_peak_post'] for session in sessions[track_type]]
        stack = np.stack(all_sessions, axis=-1)
        return np.mean([norm(stack[...,i]) for i in range(stack.shape[-1])], axis=0)
    elif pool == "sessions":
        rats = {}
        for rat,sessions in peak_posterior.items():
            stack = np.stack([session['dist_of_peak_post'] for session in sessions[track_type]], axis=-1)
            rats[rat] = np.mean([norm(stack[...,i]) for i in range(stack.shape[-1])], axis=0)
        return rats

#results_dir = '/project/bioinformatics/WZhang_lab/shared/theta_momentum/10ms_theta_nonflat/'
results_dir = '/project/bioinformatics/WZhang_lab/shared/theta_momentum/10ms_theta/'


In [ ]:
session_data = load_model_type(
    results_dir, 
    "MAP",
    "Linear",
    "preprocessed_data.pkl.bz",
    "model_theta_results.pkl.bz",
)

In [ ]:
peak_posteriors = run_peak_posteriors(session_data, "MAP")
normed_rats = pool_posteriors(peak_posteriors, pool="rats")
normed_sessions = pool_posteriors(peak_posteriors, pool="sessions")

hsepl.plot_across_session_decoding_hist(
    hsea.max_probability([normed_rats])[0],
    normed_rats
)
for rat,session in normed_sessions.items():
    hsepl.plot_across_session_decoding_hist(
        hsea.max_probability([session])[0],
        session
    )

In [ ]:
model_list = ['Diffusion', 'DiffusionScaled', 'Momentum', 'MomentumVelocityTrue', 'MomentumVelocityBias', 'CANNDynamics', 'CANNDynamicsBias', 'OUDynamicsBias', 'OUDynamicsNoBias', 'OUDynamicsScaled', 'OUDynamicsScaledNoBias']

for model_key in model_list:
    session_data = load_model_type(
        results_dir, 
        model_key,
        "Linear",
        "preprocessed_data.pkl.bz",
        "model_theta_results.pkl.bz",
    )
    peak_posteriors = run_peak_posteriors(session_data, "Gaussian")
    normed_rats = pool_posteriors(peak_posteriors, "rats")
    normed_sessions = pool_posteriors(peak_posteriors, "sessions")

    hsepl.plot_across_session_decoding_hist(
        hsea.max_probability([normed_rats])[0],
        normed_rats
    )
    for rat,session in normed_sessions:
        hsepl.plot_across_session_decoding_hist(
            hsea.max_probability([session])[0],
            session
        )
        